# Análisis Exploratorio de Datos de Sensores
## Detección de Enfermedades por Olor

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal

import sys
sys.path.append('../src')

from data.preprocessing import SensorPreprocessor
from data.feature_extraction import FeatureExtractor

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Cargar Datos

In [ ]:
# Cargar datos de ejemplo
data = np.load('../data/processed/diabetes_train.npy', allow_pickle=True).item()
signals = data['signals']
labels = data['labels']

print(f"Shape de señales: {signals.shape}")
print(f"Shape de etiquetas: {labels.shape}")
print(f"Clases únicas: {np.unique(labels)}")

## 2. Visualización de Señales Crudas

In [ ]:
# Visualizar señales de diferentes clases
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Señales de Sensores - Clase 0 (Sano)', fontsize=16)

sample_idx = np.where(labels == 0)[0][0]
sample_signals = signals[sample_idx]

for i, ax in enumerate(axes.flat):
    ax.plot(sample_signals[i])
    ax.set_title(f'Sensor {i+1}')
    ax.set_xlabel('Tiempo')
    ax.set_ylabel('Respuesta')

plt.tight_layout()
plt.show()

In [ ]:
# Comparar señales entre clases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clase 0
sample_0 = signals[np.where(labels == 0)[0][0]][0]
axes[0].plot(sample_0)
axes[0].set_title('Clase 0: Sano')
axes[0].set_xlabel('Tiempo')
axes[0].set_ylabel('Respuesta Sensor 1')

# Clase 1
sample_1 = signals[np.where(labels == 1)[0][0]][0]
axes[1].plot(sample_1, color='red')
axes[1].set_title('Clase 1: Diabetes')
axes[1].set_xlabel('Tiempo')
axes[1].set_ylabel('Respuesta Sensor 1')

plt.tight_layout()
plt.show()

## 3. Preprocesamiento

In [ ]:
preprocessor = SensorPreprocessor(sampling_rate=100)

# Procesar una muestra
sample = signals[0]
processed = preprocessor.process_sensor_array(sample)

# Visualizar antes y después
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(sample[0])
axes[0].set_title('Señal Original')
axes[0].set_ylabel('Amplitud')

axes[1].plot(processed[0])
axes[1].set_title('Señal Procesada')
axes[1].set_xlabel('Tiempo')
axes[1].set_ylabel('Amplitud Normalizada')

plt.tight_layout()
plt.show()

## 4. Análisis de Frecuencia

In [ ]:
# FFT de señales
sample_signal = processed[0]
fft = np.fft.fft(sample_signal)
freqs = np.fft.fftfreq(len(sample_signal), 1/100)

plt.figure(figsize=(12, 5))
plt.plot(freqs[:len(freqs)//2], np.abs(fft)[:len(fft)//2])
plt.title('Espectro de Frecuencia')
plt.xlabel('Frecuencia (Hz)')
plt.ylabel('Magnitud')
plt.grid(True)
plt.show()

## 5. Extracción de Características

In [ ]:
extractor = FeatureExtractor(n_pca_components=10)

# Extraer características de todas las muestras
features_list = []
for sample in signals[:100]:  # Primeras 100 muestras
    features = extractor.extract_all_features(sample)
    features_list.append(features)

features_matrix = np.array(features_list)
print(f"Shape de matriz de características: {features_matrix.shape}")

In [ ]:
# Visualizar distribución de características
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, ax in enumerate(axes.flat):
    if i < features_matrix.shape[1]:
        ax.hist(features_matrix[:, i], bins=30, alpha=0.7)
        ax.set_title(f'Feature {i+1}')
        ax.set_xlabel('Valor')
        ax.set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## 6. Análisis de Correlación

In [ ]:
# Matriz de correlación entre sensores
sample_data = signals[:50].reshape(50, -1)
correlation_matrix = np.corrcoef(sample_data.T)

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix[:8, :8], annot=True, cmap='coolwarm', center=0)
plt.title('Correlación entre Sensores')
plt.show()

## 7. Conclusiones

- Las señales muestran patrones distintivos entre clases
- El preprocesamiento mejora la calidad de la señal
- Existen características discriminativas en dominio temporal y frecuencial
- Los sensores muestran diferentes niveles de correlación